# Part 1 — Work IQ API concepts

**Microsoft IQ Deep Dive: Work IQ (Day 2)**

Work IQ is Microsoft 365's AI-native *work intelligence* layer. It answers questions and takes
action over the **signed-in user's own** work context — email, meetings, files, chats, people —
always honoring Microsoft 365 permissions and sensitivity labels.

| Concept | Value |
|---|---|
| Gateway | `https://workiq.svc.cloud.microsoft` |
| Token audience | `api://workiq.svc.cloud.microsoft` |
| Delegated scope | `WorkIQAgent.Ask` + `offline_access` |
| Work IQ app ID | `fdcc1f02-fc51-4226-8753-f668596af7f7` |
| Protocols | A2A, MCP, REST |
| Per user | Microsoft 365 Copilot license |

> Prereqs: complete `ADMIN_SETUP.md` and set `ENTRA_APP_ID` / `ENTRA_TENANT_ID` in `.env`.

## 1. Authenticate as the signed-in user

Work IQ has **no application-only mode** — every request is delegated.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))
from notebooks._shared import get_user_token, ask, call_mcp, WORK_IQ_GATEWAY

token = get_user_token()
print("Got a delegated Work IQ token:", token[:16], "...")

## 2. `ask` — natural-language over your work context

The simplest entry point: ask a question, get a grounded answer with citations to M365 items.

In [ ]:
# `ask` is a Work IQ MCP tool (JSON-RPC over the /mcp endpoint) - there is no REST chat route.
resp = ask(token, "Summarize my 5 most recent emails - who sent each and what they need from me.")
print(resp["result"]["structuredContent"]["answer"])

## 3. The context + tool API surface

Beyond `ask`, Work IQ exposes generic **verbs over M365 paths (nouns)**:

- `search_paths` — discover which M365 resource paths are available
- `get_schema` — fetch the OpenAPI schema for a path at runtime
- `fetch` — read data (e.g. `/me/messages`)
- `do_action` — trigger a side effect (e.g. `/me/sendMail`) — **the only write path**

The tool surface stays small; new data sources appear as new *paths*, not new tools.
We exercise these through MCP in Part 3 and actions in Part 4.

In [ ]:
# Discover available M365 resource paths (regex `filter`) via the search_paths MCP tool.
# search_paths returns structuredContent.paths: a list of {path, operations}.
resp = call_mcp(token, "tools/call", {"name": "search_paths", "arguments": {"filter": "mail"}})
for p in resp["result"]["structuredContent"]["paths"][:8]:
    print(p["path"], "->", ", ".join(p["operations"]))

### Recap

- Work IQ = agentic, comprehensive, secure work intelligence over M365.
- Delegated-only auth; Copilot license required.
- Small verb set over runtime-discovered paths.

**Next:** Part 2 — the A2A agent card and JSON-RPC.